In [1]:
%uv pip install torch triton

Using Python 3.12.6 environment at: /usr/local
Audited 2 packages in 19ms
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

import triton
import triton.language as tl

In [9]:
@triton.jit
def rms_norm_k(
    x_ptr,
    weight_ptr,
    output_ptr,
    n_cols:tl.constexpr,
    eps:tl.constexpr,
    BLOCK_SIZE:tl.constexpr
):
    row = tl.program_id(0)

    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    row_start = row*n_cols
    offs = row_start + cols

    x = tl.load(x_ptr + offs, mask=mask, other=0.0).to(tl.float32)

    sum_sqr = tl.sum(x*x, axis=0)
    mean_sqr = sum_sqr / n_cols
    reciprocal_rms = tl.rsqrt(mean_sqr+eps)

    weight = tl.load(weight_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    output = x * reciprocal_rms * weight

    tl.store(
        output_ptr + offs,
        output,
        mask=mask
    )

In [10]:
def rms_norm_triton(x, weight, eps=1e-6):
    if not x.is_cuda or not weight.is_cuda:
        raise ValueError("x and weight must be CUDA tensors")

    if weight.ndim != 1 or weight.shape[0] != x.shape[-1]:
        raise ValueError("weight must have shape (x.shape[-1],)")

    n_cols = x.shape[-1]

    x_2d = x.contiguous().view(-1, n_cols)
    weight = weight.contiguous()

    n_rows = x_2d.shape[0]
    output = torch.empty_like(x_2d)

    block_size = triton.next_power_of_2(n_cols)

    grid = (n_rows,)

    rms_norm_k[grid](
        x_2d, 
        weight,
        output,
        n_cols,
        eps,
        BLOCK_SIZE=block_size,
        num_warps=4
    )

    return output.view_as(x)

In [11]:
DEVICE = triton.runtime.driver.active.get_active_torch_device()

widths = [
    1,
    3,
    31,
    32,
    33,
    127,
    128,
    129,
    256,
    511,
    512,
    1000,
    1024,
    2048,
    4096,
    8192,
]

dtypes = [torch.float32, torch.float16]

if torch.cuda.is_bf16_supported():
    dtypes.append(torch.bfloat16)

torch.manual_seed(0)

for dtype in dtypes:
    if dtype == torch.float32:
        atol, rtol = 1e-5, 1e-4
    else:
        atol, rtol = 1e-2, 1e-2

    for n_cols in widths:
        x = torch.randn(
            (17, n_cols),
            device=DEVICE,
            dtype=dtype,
        )

        weight = torch.randn(
            n_cols,
            device=DEVICE,
            dtype=dtype,
        )

        actual = rms_norm_triton(x, weight, eps=1e-6)

        expected = torch.nn.functional.rms_norm(
            x,
            normalized_shape=(n_cols,),
            weight=weight,
            eps=1e-6,
        )

        torch.testing.assert_close(
            actual,
            expected,
            atol=atol,
            rtol=rtol,
        )

        max_error = (actual.float() - expected.float()).abs().max()

        print(
            f"dtype={str(dtype):14s} "
            f"width={n_cols:5d} "
            f"block={triton.next_power_of_2(n_cols):5d} "
            f"max_error={max_error.item():.3e}"
        )

dtype=torch.float32  width=    1 block=    1 max_error=0.000e+00
dtype=torch.float32  width=    3 block=    4 max_error=1.192e-07
dtype=torch.float32  width=   31 block=   32 max_error=4.768e-07
dtype=torch.float32  width=   32 block=   32 max_error=4.768e-07
dtype=torch.float32  width=   33 block=   64 max_error=4.768e-07
dtype=torch.float32  width=  127 block=  128 max_error=4.768e-07
dtype=torch.float32  width=  128 block=  128 max_error=4.768e-07
dtype=torch.float32  width=  129 block=  256 max_error=4.768e-07
dtype=torch.float32  width=  256 block=  256 max_error=4.768e-07
dtype=torch.float32  width=  511 block=  512 max_error=9.537e-07
dtype=torch.float32  width=  512 block=  512 max_error=7.153e-07
dtype=torch.float32  width= 1000 block= 1024 max_error=9.537e-07
dtype=torch.float32  width= 1024 block= 1024 max_error=9.537e-07
dtype=torch.float32  width= 2048 block= 2048 max_error=1.431e-06
dtype=torch.float32  width= 4096 block= 4096 max_error=1.431e-06
dtype=torch.float32  widt

In [12]:
edge_cases = {
    "zeros": torch.zeros(
        (4, 33),
        device=DEVICE,
        dtype=torch.float16,
    ),
    "large": torch.full(
        (4, 33),
        300.0,
        device=DEVICE,
        dtype=torch.float16,
    ),
    "tiny": torch.full(
        (4, 33),
        1e-4,
        device=DEVICE,
        dtype=torch.float16,
    ),
}

weight = torch.ones(
    33,
    device=DEVICE,
    dtype=torch.float16,
)

for name, x in edge_cases.items():
    actual = rms_norm_triton(x, weight, eps=1e-6)

    expected = torch.nn.functional.rms_norm(
        x,
        normalized_shape=(33,),
        weight=weight,
        eps=1e-6,
    )

    torch.testing.assert_close(
        actual,
        expected,
        atol=1e-2,
        rtol=1e-2,
    )

    print(
        f"{name:5s}: "
        f"finite={torch.isfinite(actual).all().item()}, "
        f"max_error={(actual.float() - expected.float()).abs().max().item():.3e}"
    )

zeros: finite=True, max_error=0.000e+00
large: finite=True, max_error=0.000e+00
tiny : finite=True, max_error=0.000e+00
